# 📱 Read the vibe: why LSTMs remember (RNN vs LSTM)

Same text, two brains. A **vanilla RNN** and an **LSTM** read real human writing and try to guess the vibe. Spoiler: one of them has a memory problem 🫠. By the end you'll know exactly *why*, and you'll have built the fix with your own hands.

Pick your vibe check in the settings cell 👇
- **`goemotions`**: Reddit comments → 7 emotions (anger, disgust, fear, joy, sadness, surprise, neutral)
- **`tweet_emoji`**: tweets → which of 20 emojis fits 🔥😂❤
- **`imdb`**: long movie reviews → positive / negative

**What you'll do**
1. Load + explore the data 🔍
2. Turn text into numbers (done for you, but read it!)
3. ✍️ TODO: write an **LSTM cell by hand** (`lstm_gates`, `MyLSTMCell`) and prove it matches PyTorch's `nn.LSTMCell`
4. ✍️ TODO: build `SequenceClassifier` (embedding → RNN/LSTM → head)
5. ✍️ TODO: write the training loop + accuracy
6. ✍️ TODO: `init_forget_bias`, the one-line trick that makes LSTMs remember from step 1
7. 🏁 Race: RNN vs LSTM vs LSTM + forget bias 1
8. 🛣️ Measure the **vanishing gradient** yourself: how much gradient reaches each word?
9. 🎤 Bonus: type your own sentence, then 🚀 submit to DL Lab

**How to run:** `Runtime → Change runtime type → T4 GPU`, fill the settings, then `Runtime → Run all`. Every `# TODO` has a ✅ check cell right after it. If a check fails, read the hint, fix, re-run.

In [ ]:
#@title ⚙️ Settings: paste your platform details, pick a dataset
PLATFORM_URL = ""   #@param {type:"string"}
SUBMIT_TOKEN = ""   #@param {type:"string"}
DATASET = "goemotions"  #@param ["goemotions", "tweet_emoji", "imdb"]
QUICK = False  #@param {type:"boolean"}

In [ ]:
#@title 🧰 Setup: install `datasets`, imports, pick a device
!pip install -q datasets

import copy, random, re, time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    # Apple's MPS is skipped on purpose: recurrent layers run slower there than on the CPU
    return torch.device("cpu")

device = pick_device()
print("🖥️ device:", device, "| torch", torch.__version__, "| dataset:", DATASET, "| QUICK:", QUICK)
if device.type != "cuda":
    print("⚠️ No GPU here. On Colab: Runtime → Change runtime type → T4 GPU (then re-run).")
    if not QUICK:
        print("   Or tick QUICK in the settings cell for a tiny CPU-friendly run.")

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1. 🔍 Load the data

| `DATASET` | Source (Hugging Face) | Task | Max tokens |
|---|---|---|---|
| `goemotions` | `google-research-datasets/go_emotions` (Apache-2.0) | Reddit comment → 1 of 7 Ekman emotions | 40 |
| `tweet_emoji` | `cardiffnlp/tweet_eval`, config `emoji` | tweet → 1 of 20 emojis | 40 |
| `imdb` | `stanfordnlp/imdb` | movie review → pos / neg | 200 |

- **goemotions** originally has 28 fine-grained labels and comments can have several. We keep comments with **exactly one** label and map the 28 labels onto the 7 **Ekman** groups with the official mapping (it's the `EKMAN` dict below).
- **tweet_emoji**: the dataset's license is listed as *unknown*, so treat it as **classroom use only** 🏫.
- **imdb**: reviews are *long* (hundreds of words). That's exactly where the vanilla RNN's **vanishing gradient** hurts: the signal from early words has to survive ~200 multiplications to reach the end. Watch for it! 👀 (imdb has no validation split, so we carve 10% off train.)

We use the official train / validation / test splits. With `QUICK=True` we only take a small random subset of each.

In [ ]:
# Official GoEmotions → Ekman mapping (28 labels → 6 Ekman emotions + neutral)
EKMAN = {
    "anger":    ["anger", "annoyance", "disapproval"],
    "disgust":  ["disgust"],
    "fear":     ["fear", "nervousness"],
    "joy":      ["joy", "amusement", "approval", "excitement", "gratitude", "love", "optimism",
                 "relief", "pride", "admiration", "desire", "caring"],
    "sadness":  ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral":  ["neutral"],
}

SUBSET = dict(train=3000, val=500, test=1000) if QUICK else dict(train=None, val=None, test=None)

def take(ds, n):
    """Random subset of n examples (shuffle first: imdb is sorted by label!)."""
    if n is not None and len(ds) > n:
        ds = ds.shuffle(seed=SEED).select(range(n))
    return ds

def clean(text):
    return text.replace("<br />", " ")   # imdb has HTML line breaks

if DATASET == "goemotions":
    raw = load_dataset("google-research-datasets/go_emotions", "simplified")
    fine_names = raw["train"].features["labels"].feature.names
    LABEL_NAMES = list(EKMAN)
    fine_to_ekman = {fine_names.index(f): LABEL_NAMES.index(g) for g, fs in EKMAN.items() for f in fs}
    assert len(fine_to_ekman) == 28
    splits = {"train": raw["train"], "val": raw["validation"], "test": raw["test"]}
    splits = {k: v.filter(lambda ex: len(ex["labels"]) == 1) for k, v in splits.items()}   # single-label only
    get_label = lambda ex: fine_to_ekman[ex["labels"][0]]
    MAX_LEN, TASK = 40, "emotion"
elif DATASET == "tweet_emoji":
    raw = load_dataset("cardiffnlp/tweet_eval", "emoji")
    LABEL_NAMES = raw["train"].features["label"].names
    splits = {"train": raw["train"], "val": raw["validation"], "test": raw["test"]}
    get_label = lambda ex: ex["label"]
    MAX_LEN, TASK = 40, "emoji"
elif DATASET == "imdb":
    raw = load_dataset("stanfordnlp/imdb")
    LABEL_NAMES = ["negative", "positive"]
    carved = raw["train"].train_test_split(test_size=0.1, seed=SEED, stratify_by_column="label")
    splits = {"train": carved["train"], "val": carved["test"], "test": raw["test"]}
    get_label = lambda ex: ex["label"]
    MAX_LEN, TASK = 200, "sentiment"
else:
    raise ValueError(f"Unknown DATASET {DATASET!r}")

data = {}
for name, ds in splits.items():
    ds = take(ds, SUBSET[name])
    data[name] = ([clean(t) for t in ds["text"]], [get_label(ex) for ex in ds])

train_texts, train_labels = data["train"]
val_texts, val_labels = data["val"]
test_texts, test_labels = data["test"]
NUM_CLASSES = len(LABEL_NAMES)
print(f"train {len(train_texts):,} | val {len(val_texts):,} | test {len(test_texts):,} | {NUM_CLASSES} classes")

In [ ]:
#@title 👀 Peek at the data + class balance
rng = random.Random(1)
for i in rng.sample(range(len(train_texts)), 5):
    t = train_texts[i]
    print(f"[{LABEL_NAMES[train_labels[i]]}]  {t[:160]}{'…' if len(t) > 160 else ''}")

counts = Counter(train_labels)
print("\nClass balance (train):")
for c in range(NUM_CLASSES):
    print(f"  {c:2d} {LABEL_NAMES[c]:>10s}  {counts[c]:6,}  ({100 * counts[c] / len(train_labels):5.1f}%)")
majority = counts.most_common(1)[0][0]
BASELINE = sum(y == majority for y in test_labels) / len(test_labels)
print(f"\n🎯 'Always guess {LABEL_NAMES[majority]}' gets {100 * BASELINE:.1f}% test accuracy. Your models must beat this!")

# emoji glyphs often don't render in matplotlib fonts → use indices for tweet_emoji
tick_names = [str(c) for c in range(NUM_CLASSES)] if DATASET == "tweet_emoji" else LABEL_NAMES
plt.figure(figsize=(8, 3))
plt.bar(tick_names, [counts[c] for c in range(NUM_CLASSES)], color="#7c5cff")
plt.title(f"{DATASET}: train label counts"); plt.ylabel("examples")
plt.xticks(rotation=45 if NUM_CLASSES > 2 else 0); plt.tight_layout(); plt.show()
if DATASET == "tweet_emoji":
    print("label index → emoji:", "  ".join(f"{i}:{e}" for i, e in enumerate(LABEL_NAMES)))

## 2. 🔢 Text → numbers (provided)

Neural nets eat numbers, not words. The recipe:
1. **Tokenise**: lowercase, then grab words with a regex (`[a-z0-9']+`) and treat each punctuation mark / emoji as its own token.
2. **Vocab**: keep the most common words from the *train* split. Index `0 = <pad>`, `1 = <unk>` (any word we never saw).
3. **Encode + truncate** each text to at most `MAX_LEN` ids.
4. **Pad** each batch to the length of its longest sequence with `<pad>`, and remember the real `lengths`. 🧠 Remember those lengths, you'll need them in the model!

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9']+|[^\sa-z0-9']")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

VOCAB_SIZE = 20_000 if DATASET == "imdb" else 15_000
PAD, UNK = 0, 1
word_counts = Counter(tok for t in train_texts for tok in tokenize(t))
itos = ["<pad>", "<unk>"] + [w for w, c in word_counts.most_common(VOCAB_SIZE - 2) if c >= 2]
stoi = {w: i for i, w in enumerate(itos)}

def encode(text):
    ids = [stoi.get(tok, UNK) for tok in tokenize(text)][:MAX_LEN]
    return ids or [UNK]          # never return an empty sequence

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.ids = [encode(t) for t in texts]
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return self.ids[i], self.labels[i]

def collate(batch):
    """List of (ids, label) → x (B, T) padded with PAD, lengths (B,), y (B,)."""
    lengths = torch.tensor([len(ids) for ids, _ in batch])
    x = torch.full((len(batch), int(lengths.max())), PAD, dtype=torch.long)
    for row, (ids, _) in enumerate(batch):
        x[row, :len(ids)] = torch.tensor(ids)
    y = torch.tensor([label for _, label in batch])
    return x, lengths, y

BATCH_SIZE = 64
train_ds, val_ds, test_ds = TextDataset(train_texts, train_labels), TextDataset(val_texts, val_labels), TextDataset(test_texts, test_labels)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=256, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=256, collate_fn=collate)

print(f"vocab size: {len(itos):,}")
print("example:", tokenize(train_texts[0])[:15])
print("ids:    ", encode(train_texts[0])[:15])
x, lengths, y = next(iter(train_loader))
print("one batch → x", tuple(x.shape), "| lengths", lengths[:8].tolist(), "… | y", y[:8].tolist(), "…")

raw_lens = [len(tokenize(t)) for t in train_texts]
plt.figure(figsize=(7, 2.8))
plt.hist(raw_lens, bins=50, color="#22c55e"); plt.axvline(MAX_LEN, color="k", ls="--", label=f"MAX_LEN={MAX_LEN}")
plt.title(f"tokens per text ({100 * np.mean(np.array(raw_lens) > MAX_LEN):.0f}% get truncated)"); plt.legend(); plt.show()

## 3. ✍️ TODO: an LSTM cell, by hand

On the platform you did `rnn_step` and `lstm_cell` in NumPy. Now the same thing in PyTorch, so we can line it up against the real `nn.LSTMCell`.

A vanilla RNN has **one** state `h` and squashes it through `tanh` every step. The LSTM keeps a second state, the **cell state `c`** (its long-term memory 🧠), and three gates that decide what happens to it:

| gate | activation | job |
|---|---|---|
| **i** input | σ (0…1) | how much new stuff to *write* |
| **f** forget | σ (0…1) | how much old memory to *keep* |
| **g** candidate | tanh (−1…1) | *what* new stuff to write |
| **o** output | σ (0…1) | how much memory to *show* as `h` |

```
z = W·x + U·h + b                 # one big (B, 4H) pre-activation
i, f, g, o = split(z, 4)          # PyTorch order: input, forget, candidate(g), output
c_new = f ⊙ c + i ⊙ g             # keep some old memory, add some new
h_new = o ⊙ tanh(c_new)           # show some of it
```

Note the `+` in `c_new`: memory is **added to**, not squashed through a weight matrix. Hold that thought, it's the whole reason LSTMs fix the vanishing gradient (section 8).

Our `MyLSTMCell` uses two linear layers, `W` (input → 4H) and `U` (hidden → 4H), exactly like PyTorch's `weight_ih` / `weight_hh`, with the same gate order. So we can copy PyTorch's weights straight in and the outputs must match.

Fill in the blanks:
1. `lstm_gates`: sigmoid for `i`, `f`, `o`; tanh for `g`
2. `c_new` and `h_new` in `MyLSTMCell.forward`

In [ ]:
def lstm_gates(z):
    """z: (B, 4H) pre-activations in PyTorch order [input, forget, candidate g, output] → activated (i, f, g, o)."""
    i, f, g, o = z.chunk(4, dim=1)          # four (B, H) pieces
    i = ...  # TODO: input gate in (0, 1)
    f = ...  # TODO: forget gate in (0, 1)
    g = ...  # TODO: candidate in (-1, 1)
    o = ...  # TODO: output gate in (0, 1)
    return i, f, g, o

class MyLSTMCell(nn.Module):
    """One LSTM step, written out by hand. Same weights layout + gate order as nn.LSTMCell."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W = nn.Linear(input_size, 4 * hidden_size)    # like nn.LSTMCell.weight_ih / bias_ih
        self.U = nn.Linear(hidden_size, 4 * hidden_size)   # like nn.LSTMCell.weight_hh / bias_hh

    def forward(self, x, state):
        h, c = state                                        # each (B, H)
        i, f, g, o = lstm_gates(self.W(x) + self.U(h))
        c_new = ...  # TODO: forget some old memory, write some new (the additive update!)
        h_new = ...  # TODO: output gate × squashed memory
        return h_new, c_new

In [ ]:
#@title ✅ Check your LSTM cell against PyTorch's nn.LSTMCell
torch.manual_seed(0)
z = torch.randn(3, 4 * 5) * 3
gi, gf, gg, go = lstm_gates(z)
assert all(isinstance(t, torch.Tensor) and t.shape == (3, 5) for t in (gi, gf, gg, go)), "lstm_gates should return four (B, H) tensors (is one of i, f, g, o still a `...` blank?)"
for name, t in [("i", gi), ("f", gf), ("o", go)]:
    assert t.min() > 0 and t.max() < 1, f"gate {name} must be in (0, 1). Hint: torch.sigmoid"
assert gg.min() < 0, "the candidate g must be able to go negative. Hint: torch.tanh, not sigmoid"
assert torch.allclose(gg, torch.tanh(z[:, 10:15])), "g should be tanh of the THIRD chunk (order: i, f, g, o)"

I, H = 6, 5
ref = nn.LSTMCell(I, H)
mine = MyLSTMCell(I, H)
with torch.no_grad():                                       # copy PyTorch's weights into ours
    mine.W.weight.copy_(ref.weight_ih); mine.W.bias.copy_(ref.bias_ih)
    mine.U.weight.copy_(ref.weight_hh); mine.U.bias.copy_(ref.bias_hh)
h, c = torch.randn(4, H), torch.randn(4, H)
h_ref, c_ref = h.clone(), c.clone()
for t in range(10):                                         # 10 steps, errors would pile up
    x_t = torch.randn(4, I)
    h, c = mine(x_t, (h, c))
    h_ref, c_ref = ref(x_t, (h_ref, c_ref))
    assert isinstance(h, torch.Tensor) and h.shape == (4, H), "MyLSTMCell should return (h_new, c_new), each (B, H)"
    assert torch.allclose(c, c_ref, atol=1e-6), f"step {t}: c_new differs from nn.LSTMCell. Hint: c_new = f * c + i * g"
    assert torch.allclose(h, h_ref, atol=1e-6), f"step {t}: h_new differs from nn.LSTMCell. Hint: h_new = o * tanh(c_new)"
print(f"✅ nice: your cell matches nn.LSTMCell for 10 steps (max diff {(h - h_ref).abs().max().item():.1e})")

### 🐢 vs 🚀 Same maths, different speed

`nn.LSTM` runs exactly your cell, looped over time, but in fused CUDA (cuDNN) kernels. Let's unroll **your** cell over a batch of real sentences with a Python `for` loop and compare. Same answer, but slower (a lot slower on a GPU, where one fused kernel beats hundreds of tiny Python-launched ones; on a CPU the gap is smaller), so for training we'll use `nn.LSTM` (and `nn.RNN`) from here on. Everything you learn about your cell still holds 1:1.

In [ ]:
#@title 🐢 Your cell, unrolled over time, vs nn.LSTM
torch.manual_seed(0)
B_demo, E_demo, H_demo = 64, 64, 128
fast = nn.LSTM(E_demo, H_demo, batch_first=True).to(device)
slow = MyLSTMCell(E_demo, H_demo).to(device)
with torch.no_grad():   # nn.LSTM's layer-0 weights have the same layout as nn.LSTMCell's
    slow.W.weight.copy_(fast.weight_ih_l0); slow.W.bias.copy_(fast.bias_ih_l0)
    slow.U.weight.copy_(fast.weight_hh_l0); slow.U.bias.copy_(fast.bias_hh_l0)
x_demo = torch.randn(B_demo, MAX_LEN, E_demo, device=device)

def unroll(cell, x):
    h = c = torch.zeros(x.size(0), cell.hidden_size, device=x.device)
    outs = []
    for t in range(x.size(1)):        # one Python step per token 🐢
        h, c = cell(x[:, t], (h, c))
        outs.append(h)
    return torch.stack(outs, dim=1)   # (B, T, H), same as nn.LSTM's `out`

def timeit(fn, reps=5):
    fn()
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps):
        out = fn()
    if device.type == "cuda": torch.cuda.synchronize()
    return out, (time.time() - t0) / reps

with torch.no_grad():
    out_slow, t_slow = timeit(lambda: unroll(slow, x_demo))
    out_fast, t_fast = timeit(lambda: fast(x_demo)[0])
print(f"max |difference| over {MAX_LEN} steps: {(out_slow - out_fast).abs().max().item():.1e}  (same maths ✅)")
print(f"your cell in a Python loop: {1000 * t_slow:6.1f} ms   |   nn.LSTM: {1000 * t_fast:6.1f} ms   →  nn.LSTM is {t_slow / t_fast:.1f}× faster")

## 4. ✍️ TODO: the classifier

Now wrap the recurrent layer into a text classifier: tokens → embedding → RNN or LSTM → **hidden state at the last real token** → linear head → logits.

| `cell` | PyTorch | One step (what you coded) |
|---|---|---|
| `"rnn"` | `nn.RNN` | `rnn_step`: h' = tanh(Wx·x + Wh·h + b) |
| `"lstm"` | `nn.LSTM` | `lstm_cell` / `MyLSTMCell`: c' = f⊙c + i⊙g, h' = o⊙tanh(c') |

With `batch_first=True`, `out, _ = self.rnn(emb)` gives `out` of shape **(B, T, H)**: the hidden state after **every** step.

🚨 **The padding trap:** `out[:, -1]` is the state after the last *column*, and for short texts that column is `<pad>`. The model has been chewing on padding! We want the state at the **last real token**, position `lengths - 1` in each row.
Hint: advanced indexing `out[rows, cols]` with `rows = torch.arange(B)` and `cols = lengths - 1` picks one time step per row → shape (B, H).

(The fancy alternative is `nn.utils.rnn.pack_padded_sequence`; indexing is simpler and gives the same answer for a one-direction RNN.)

Fill in the three blanks:
1. pick the RNN class for `cell` (hint: the `RNN_TYPES` dict)
2. the head: a linear layer from `hidden` to `num_classes`
3. `h_last`: the hidden state at the last real token

In [ ]:
RNN_TYPES = {"rnn": nn.RNN, "lstm": nn.LSTM}

class SequenceClassifier(nn.Module):
    """tokens → embedding → RNN / LSTM → hidden state at the last real token → linear head → logits"""

    def __init__(self, vocab_size, num_classes, cell="lstm", emb_dim=128, hidden=128):
        super().__init__()
        self.cell = cell
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        rnn_cls = ...  # TODO: pick nn.RNN / nn.LSTM for this `cell`
        self.rnn = rnn_cls(emb_dim, hidden, batch_first=True)
        self.head = ...  # TODO: linear layer hidden → num_classes

    def forward(self, x, lengths):
        # x: (B, T) token ids, lengths: (B,) number of real (non-pad) tokens per row
        emb = self.embedding(x)                    # (B, T, E)
        out, _ = self.rnn(emb)                     # (B, T, H) hidden state after every step
        lengths = lengths.to(out.device)
        rows = torch.arange(out.size(0), device=out.device)
        h_last = ...  # TODO: (B, H) hidden state at the LAST REAL token of each row
        return self.head(h_last)                   # (B, num_classes) logits

In [ ]:
#@title ✅ Check your model
torch.manual_seed(0)
for cell, cls in [("rnn", nn.RNN), ("lstm", nn.LSTM)]:
    m = SequenceClassifier(50, 5, cell=cell, emb_dim=8, hidden=16)
    assert isinstance(m.rnn, cls), f"cell='{cell}' should give self.rnn = {cls.__name__}, got {type(m.rnn).__name__}. Hint: RNN_TYPES[cell]"
    assert isinstance(m.head, nn.Linear) and (m.head.in_features, m.head.out_features) == (16, 5), \
        "self.head should be nn.Linear(hidden, num_classes)"
    x = torch.randint(2, 50, (3, 7)); lengths = torch.tensor([7, 4, 2])
    x[1, 4:] = PAD; x[2, 2:] = PAD
    logits = m(x, lengths)
    assert logits.shape == (3, 5), f"logits should be (batch, num_classes) = (3, 5), got {tuple(logits.shape)}"
    x_more_pad = torch.cat([x, torch.full((3, 5), PAD)], dim=1)
    assert torch.allclose(m(x_more_pad, lengths), logits, atol=1e-5), \
        "Extra padding changed the output 😬. Hint: take out[rows, lengths - 1], not out[:, -1]."
    assert torch.allclose(m(x[2:3, :2], lengths[2:3]), logits[2:3], atol=1e-5), \
        "A short text should give the same logits alone as inside a padded batch."
print("✅ nice: both cells build, give (B, num_classes) logits, and ignore padding")

## 5. ✍️ TODO: train + evaluate

The classic five-step training dance, once per batch:
1. **forward**: `logits = model(x, lengths)`
2. **loss**: cross-entropy between `logits` and the true labels `y`
3. **`zero_grad`**: wipe old gradients (PyTorch *adds* them up otherwise)
4. **`backward`**: backprop through time 🔙
5. **`clip_grad_norm_`** then **`step`**: clipping caps the gradient norm so RNNs don't *explode* (it can't fix *vanishing* though!)

Then `evaluate` = fraction of examples where the arg-max of the logits matches the label.

In [ ]:
def train_one_epoch(model, loader, optimizer, clip=1.0):
    model.train()
    total_loss, n = 0.0, 0
    for x, lengths, y in loader:
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        logits = ...  # TODO: forward pass
        loss = ...  # TODO: cross-entropy loss
        ...  # TODO: reset gradients
        ...  # TODO: backprop through time
        ...  # TODO: clip the gradient norm to `clip`
        ...  # TODO: update the weights
        total_loss += loss.item() * len(y)
        n += len(y)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader):
    """Accuracy in [0, 1]."""
    model.eval()
    correct, total = 0, 0
    for x, lengths, y in loader:
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        logits = model(x, lengths)
        preds = ...  # TODO: predicted class per example
        correct += ...  # TODO: how many predictions are right (a Python int)
        total += len(y)
    return correct / total

In [ ]:
#@title ✅ Check your training loop
torch.manual_seed(0)
toy_x = torch.randint(2, 20, (64, 6))
toy_y = (toy_x == 5).any(dim=1).long()           # toy task: "is token 5 in the sentence?"
toy_loader = DataLoader(list(zip(toy_x, torch.full((64,), 6), toy_y)), batch_size=16)

class AlwaysZero(nn.Module):
    def forward(self, x, lengths):
        return torch.tensor([[1.0, 0.0]], device=x.device).repeat(len(x), 1)
acc0 = evaluate(AlwaysZero(), toy_loader)
assert isinstance(acc0, float), f"evaluate should return a float, got {type(acc0).__name__}"
assert abs(acc0 - (toy_y == 0).float().mean().item()) < 1e-6, "evaluate: a model that always says class 0 should score the share of class-0 labels"

calls = Counter()
_real_clip = torch.nn.utils.clip_grad_norm_
def _counting_clip(*a, **k):
    calls["clip"] += 1
    return _real_clip(*a, **k)
torch.nn.utils.clip_grad_norm_ = _counting_clip
try:
    toy = SequenceClassifier(20, 2, "lstm", emb_dim=16, hidden=32).to(device)
    opt = torch.optim.Adam(toy.parameters(), lr=1e-2)
    for name in ["zero_grad", "step"]:
        real = getattr(opt, name)
        setattr(opt, name, (lambda real, name: lambda *a, **k: (calls.update([name]), real(*a, **k))[1])(real, name))
    losses = [train_one_epoch(toy, toy_loader, opt) for _ in range(25)]
finally:
    torch.nn.utils.clip_grad_norm_ = _real_clip
assert isinstance(losses[0], float), "train_one_epoch should return the average loss as a float"
assert calls["zero_grad"] == calls["step"] == calls["clip"] == 25 * 4, \
    f"expected zero_grad, clip_grad_norm_ and step once per batch, got {dict(calls)}"
assert losses[-1] < 0.5 * losses[0], f"loss should drop a lot on the toy task ({losses[0]:.3f} → {losses[-1]:.3f}). Order: zero_grad → backward → clip → step"
assert evaluate(toy, toy_loader) > 0.9, "the toy model should get >90% after training"
print(f"✅ nice: toy loss {losses[0]:.3f} → {losses[-1]:.3f}, toy accuracy {evaluate(toy, toy_loader):.0%}")

## 6. ✍️ TODO: the forget-gate bias trick

At the start of training the weights are small, so each gate is roughly **σ(its bias)**. PyTorch initialises every LSTM bias near 0, so

- forget gate at init: **f ≈ σ(0) = 0.5**
- memory left after T steps ≈ **f^T = σ(b)^T** → after 10 tokens: 0.5¹⁰ ≈ 0.001 😱

So a fresh LSTM forgets almost as fast as a vanilla RNN, and it has to *learn* to remember. And since the gradient along the cell state is multiplied by the same `f` every step (more in section 8), the gradient that would teach it to remember is tiny too. Chicken, meet egg 🐔🥚.

The fix (Gers et al. 2000, Jozefowicz et al. 2015): start the forget gate **open** by setting its bias to **1**: σ(1) ≈ 0.73, and 0.73¹⁰ ≈ 0.04, about 45× more memory after 10 tokens (and way more after 40).

PyTorch's `nn.LSTM` has **two** bias vectors per layer, `bias_ih_l0` and `bias_hh_l0`, each of shape (4H,), that just get added together. Gate order is again input, forget, g, output, so the **forget slice is `[H : 2H]`**.

✍️ Write `init_forget_bias(lstm, value)`: fill the forget slice of `bias_ih_l0` with `value` and zero the forget slice of `bias_hh_l0` (so the total forget bias is exactly `value`). Leave the other gates alone.
Hint: inside `torch.no_grad()`, `tensor[a:b].fill_(v)` and `tensor[a:b].zero_()` work in place.

In [ ]:
#@title 📉 How fast does memory fade? σ(b)^T
sig = lambda b: 1 / (1 + np.exp(-b))
T_axis = np.arange(0, MAX_LEN + 1)
plt.figure(figsize=(7, 3.5))
for b, col in [(0.0, "#94a3b8"), (1.0, "#8b5cf6"), (2.0, "#3b82f6"), (3.0, "#0ea5e9")]:
    lbl = f"b = {b:.0f}  (f = σ(b) = {sig(b):.2f})" + ("  ← PyTorch default ≈ 0" if b == 0 else "")
    plt.semilogy(T_axis, sig(b) ** T_axis, color=col, lw=2, label=lbl)
plt.xlabel("tokens later (T)"); plt.ylabel("memory left ≈ σ(b)^T")
plt.title("Forget-gate bias at init → how long the cell remembers"); plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()
for T in sorted({10, MAX_LEN}):
    print(f"after {T:3d} tokens:  b=0 → {sig(0) ** T:.1e}   b=1 → {sig(1) ** T:.1e}   ({(sig(1) / sig(0)) ** T:,.0f}× more)")

In [ ]:
def init_forget_bias(lstm, value=1.0):
    """Set the forget-gate bias of a 1-layer nn.LSTM to `value` (in place). Returns the lstm."""
    H = lstm.hidden_size
    with torch.no_grad():
        ...  # TODO: forget slice of bias_ih_l0 = value
        ...  # TODO: forget slice of bias_hh_l0 = 0
    return lstm

In [ ]:
#@title ✅ Check init_forget_bias
torch.manual_seed(0)
lstm_t = nn.LSTM(4, 6, batch_first=True)
before_ih, before_hh = lstm_t.bias_ih_l0.detach().clone(), lstm_t.bias_hh_l0.detach().clone()
init_forget_bias(lstm_t, 1.0)
H = 6
f_ih, f_hh = lstm_t.bias_ih_l0[H:2 * H], lstm_t.bias_hh_l0[H:2 * H]
assert torch.allclose(f_ih + f_hh, torch.ones(H)), \
    f"total forget bias should be 1.0 everywhere, got {(f_ih + f_hh).detach().numpy().round(2)}. Hint: the forget slice is [H:2H]"
assert torch.allclose(f_hh, torch.zeros(H)), "zero the forget slice of bias_hh_l0 so the total is exactly `value`"
for name, before, now in [("bias_ih_l0", before_ih, lstm_t.bias_ih_l0), ("bias_hh_l0", before_hh, lstm_t.bias_hh_l0)]:
    for gate, sl in [("input", slice(0, H)), ("g", slice(2 * H, 3 * H)), ("output", slice(3 * H, 4 * H))]:
        assert torch.equal(now[sl], before[sl]), f"{name}: the {gate} slice changed. Only touch the forget slice [H:2H]"
assert lstm_t.bias_ih_l0.requires_grad, "keep the bias a trainable parameter (edit it in place under torch.no_grad())"
# with x = 0 and h = 0 the forget gate is exactly σ(total forget bias) = σ(1)
f0 = torch.sigmoid(lstm_t.bias_ih_l0[H:2 * H] + lstm_t.bias_hh_l0[H:2 * H])
print(f"✅ nice: forget bias set → at init f ≈ σ(1) = {f0.mean().item():.3f} (default was ≈ 0.5)")

## 7. 🏁 The race: RNN vs LSTM vs LSTM + forget bias 1

Three runs, same data, same embedding size, same hidden size, same optimizer, same epochs, same random seed:

| run | recurrent layer | forget-gate bias at init |
|---|---|---|
| `rnn` | `nn.RNN` | (no gates) |
| `lstm` | `nn.LSTM` | PyTorch default (≈ 0 → f ≈ 0.5) |
| `lstm_fb1` | `nn.LSTM` | **1.0** via your `init_forget_bias` (f ≈ 0.73) |

The two LSTMs start from **identical** weights except for the forget bias, so any difference is the trick's doing. After each epoch we check validation accuracy and keep the best checkpoint, then score it **once** on the test set.

⏱️ On a T4 this takes roughly 10 minutes for all three (less for the short-text datasets). `QUICK=True` does 2 tiny epochs.

In [ ]:
#@title 🏋️ Train all three
EPOCHS = 2 if QUICK else {"goemotions": 10, "tweet_emoji": 6, "imdb": 10}[DATASET]
EMB, HIDDEN, LR = (64, 64, 2e-3) if QUICK else (128, 256, 2e-3)
RUNS = {"rnn": ("rnn", None), "lstm": ("lstm", None), "lstm_fb1": ("lstm", 1.0)}   # name → (cell, forget bias)
LABELS = {"rnn": "RNN", "lstm": "LSTM", "lstm_fb1": "LSTM + forget bias 1"}
COLORS = {"rnn": "#ef4444", "lstm": "#3b82f6", "lstm_fb1": "#8b5cf6"}

def make_model(cell, forget_bias=None, seed=SEED):
    torch.manual_seed(seed)
    model = SequenceClassifier(len(itos), NUM_CLASSES, cell=cell, emb_dim=EMB, hidden=HIDDEN)
    if forget_bias is not None:
        init_forget_bias(model.rnn, forget_bias)
    return model.to(device)

results = {}
for name, (cell, fb) in RUNS.items():
    model = make_model(cell, fb)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n🧠 {LABELS[name]}  ({n_params:,} params)")
    hist = {"train_loss": [], "val_acc": []}
    best_val, best_state, t0 = -1.0, None, time.time()
    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, optimizer)
        val_acc = evaluate(model, val_loader)
        hist["train_loss"].append(loss); hist["val_acc"].append(val_acc)
        if val_acc > best_val:
            best_val, best_state = val_acc, copy.deepcopy(model.state_dict())
        print(f"  epoch {epoch:2d}  train loss {loss:.3f}  val acc {100 * val_acc:5.1f}%  ({time.time() - t0:.0f}s)")
    model.load_state_dict(best_state)
    test_acc = evaluate(model, test_loader)
    results[name] = dict(model=model, hist=hist, best_val=best_val, test_acc=test_acc,
                         params=n_params, seconds=time.time() - t0)
    print(f"  ➜ best val {100 * best_val:.1f}%  |  test {100 * test_acc:.1f}%")

In [ ]:
#@title 📊 Validation curves + test accuracy
names = list(results)
best_name = max(names, key=lambda n: results[n]["test_acc"])
best_test_acc = 100 * results[best_name]["test_acc"]

def plot_val_curves(ax):
    for n in names:
        v = results[n]["hist"]["val_acc"]
        ax.plot(range(1, len(v) + 1), [100 * a for a in v], marker="o", color=COLORS[n], label=LABELS[n],
                ls="--" if n == "lstm_fb1" else "-")
    ax.axhline(100 * BASELINE, color="gray", ls=":", label="majority guess")
    ax.set_xlabel("epoch"); ax.set_ylabel("validation accuracy (%)"); ax.set_title(f"{DATASET}: validation accuracy")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

def plot_test_bars(ax):
    accs = [100 * results[n]["test_acc"] for n in names]
    bars = ax.bar([LABELS[n].replace(" + ", "\n+ ") for n in names], accs, color=[COLORS[n] for n in names])
    for b, a in zip(bars, accs):
        ax.text(b.get_x() + b.get_width() / 2, a + 0.5, f"{a:.1f}%", ha="center", fontweight="bold")
    ax.axhline(100 * BASELINE, color="gray", ls=":", label=f"majority guess ({100 * BASELINE:.1f}%)")
    ax.set_ylabel("test accuracy (%)"); ax.set_ylim(0, max(accs + [100 * BASELINE]) * 1.15)
    ax.set_title(f"{DATASET} ({TASK}): test accuracy"); ax.legend(loc="lower right", fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_val_curves(axes[0]); plot_test_bars(axes[1]); plt.tight_layout(); plt.show()

print(f"{'run':10s} {'params':>10s} {'time':>7s} {'best val':>9s} {'test':>7s}")
for n in names:
    r = results[n]
    print(f"{n:10s} {r['params']:>10,} {r['seconds']:>6.0f}s {100 * r['best_val']:>8.1f}% {100 * r['test_acc']:>6.1f}%")
gap = 100 * (results["lstm_fb1"]["hist"]["val_acc"][0] - results["lstm"]["hist"]["val_acc"][0])
print(f"\nforget bias 1 vs default after epoch 1: {gap:+.1f} points of validation accuracy")
print(f"🏆 best: {LABELS[best_name]} with {best_test_acc:.1f}% test accuracy")

## 8. 🛣️ How LSTMs solve the vanishing gradient

Training = the loss sends a "blame" signal (the gradient) **backwards** through every time step. How much blame reaches a word 30 tokens ago decides whether the model can *learn* to use that word.

**Vanilla RNN** (your `rnn_step`): `h' = tanh(Wx·x + Wh·h + b)`. Going back **one** step multiplies the gradient by

> ∂h'/∂h = diag(1 − h'²) · Wh  → a squash (≤ 1) times a weight matrix, **every single step**

Do that 30 times and the gradient shrinks *exponentially*, like photocopying a photocopy 30 times 📠. That's the **vanishing gradient**.

**LSTM** (your `MyLSTMCell`): `c' = f ⊙ c + i ⊙ g`. Along the cell state, going back one step multiplies the gradient by

> ∂c'/∂c = diag(f)  → **just the forget gate**. No weight matrix, no tanh squash.

So after k steps the gradient on the cell-state path is ≈ f₁·f₂·…·f_k. If the LSTM wants to remember (f ≈ 1), the gradient flows back almost untouched: the **gradient highway** 🛣️. (There are other, bumpier paths through `h` and the gates, but this one is the express lane.)

🧠 **The one-liner to remember:** *in an LSTM, what the cell remembers going forward (× f per step) is exactly what the gradient keeps going backward (× f per step).* Memory and gradient are the same number. That's also why the forget-bias trick works: f ≈ 0.73 instead of 0.5 at init means more memory **and** more gradient from step 1.

Clipping, by the way, only stops *exploding* gradients. It can't bring a vanished one back.

Let's measure it 🔬. We take the longest validation texts, backprop the loss, and record the gradient size that reaches the word embedding at each position. x-axis = how many tokens before the end; log-scale y-axis. Left: freshly initialised models (pure architecture). Right: the trained models. If a curve hits the floor (1e-30), the gradient has vanished so hard that float32 rounds it to exactly zero 🫠.

In [ ]:
#@title 🔬 Gradient reaching each token position (log scale)
def grad_through_time(model, x, lengths, y):
    """Average ‖∂loss/∂embedding‖ per position. Index 0 = last token, 1 = one before, …"""
    store = {}
    def hook(module, inputs, output):
        output.retain_grad(); store["emb"] = output
    handle = model.embedding.register_forward_hook(hook)
    model.train()   # cuDNN only backprops RNNs in train mode (no dropout here, so same outputs)
    model.zero_grad()
    F.cross_entropy(model(x, lengths), y).backward()
    handle.remove(); model.zero_grad()
    g = store["emb"].grad.norm(dim=2).mean(dim=0)       # (T,) average gradient size per position
    return g.flip(0).double().cpu().numpy()

order = sorted(range(len(val_ds)), key=lambda i: -len(val_ds.ids[i]))[:64]
L = min(len(val_ds.ids[i]) for i in order)               # crop everything to a common length
probe = collate([(val_ds.ids[i][:L], val_ds.labels[i]) for i in order])
x_probe, len_probe, y_probe = (t.to(device) for t in probe)

FLOOR = 1e-30   # float32 underflows to exactly 0 somewhere below this → "fully vanished"
GRAD = {"init": {}, "trained": {}}
for n, (cell, fb) in RUNS.items():
    GRAD["init"][n] = grad_through_time(make_model(cell, fb), x_probe, len_probe, y_probe)
    GRAD["trained"][n] = grad_through_time(results[n]["model"], x_probe, len_probe, y_probe)

def plot_grad(ax, which):
    for n, g in GRAD[which].items():
        rel = np.maximum(g / g[0], FLOOR)
        ax.semilogy(rel, color=COLORS[n], lw=2, label=LABELS[n], ls="--" if n == "lstm_fb1" else "-")
    ax.axhline(FLOOR, color="gray", ls=":", lw=1); ax.text(0, FLOOR * 30, "floor = vanished (float32 zero)", color="gray", fontsize=7)
    ax.set_xlabel("tokens before the end"); ax.set_ylabel("gradient at that word (relative to last word)")
    ax.set_title(f"Gradient per position, {which} (L={L})"); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_grad(axes[0], "init"); plot_grad(axes[1], "trained"); plt.tight_layout(); plt.show()

steps_back = [k for k in (5, 10, 20, 30, 50, 100, 150) if k < L]
for which in ["init", "trained"]:
    print(f"\n{which}: gradient k tokens back, relative to the last token")
    print(f"{'':22s}" + "".join(f"{k:>9d}" for k in steps_back))
    for n, g in GRAD[which].items():
        rel = g / g[0]
        print(f"{LABELS[n]:22s}" + "".join(f"{rel[k]:9.0e}" if rel[k] > FLOOR else f"{'~0':>9s}" for k in steps_back))

### 🔍 Peek inside the trained LSTMs: what are the forget gates doing?

The highway is only as good as `f`. Let's load the trained `nn.LSTM` weights into **your** `MyLSTMCell` (same layout, remember?), step through the probe texts, and record every forget-gate value. Two numbers to look at: the **mean f**, and the share of **"memory slots"** with f > 0.9 (such a slot still keeps 0.9¹⁰ ≈ 35% after 10 tokens, while an f = 0.5 slot keeps 0.1%). A model doesn't need *every* slot to remember: a few long-memory slots carrying the important stuff are enough, and those are exactly the ones the gradient highway runs through.

In [ ]:
#@title 🔍 Forget-gate values in the trained LSTMs (using your MyLSTMCell)
@torch.no_grad()
def forget_gates(lstm, emb):
    cell = MyLSTMCell(lstm.input_size, lstm.hidden_size).to(emb.device)
    cell.W.weight.copy_(lstm.weight_ih_l0); cell.W.bias.copy_(lstm.bias_ih_l0)
    cell.U.weight.copy_(lstm.weight_hh_l0); cell.U.bias.copy_(lstm.bias_hh_l0)
    h = c = torch.zeros(emb.size(0), lstm.hidden_size, device=emb.device)
    fs = []
    for t in range(emb.size(1)):
        _, f, _, _ = lstm_gates(cell.W(emb[:, t]) + cell.U(h))
        h, c = cell(emb[:, t], (h, c))
        fs.append(f)
    return torch.stack(fs, dim=1).flatten().cpu().numpy()    # every forget-gate value

plt.figure(figsize=(7, 3.5))
for n in ["lstm", "lstm_fb1"]:
    m = results[n]["model"]; m.eval()
    with torch.no_grad():
        f_all = forget_gates(m.rnn, m.embedding(x_probe))
    plt.hist(f_all, bins=50, range=(0, 1), alpha=0.6, color=COLORS[n], label=f"{LABELS[n]}  (mean f {f_all.mean():.2f})")
    print(f"{LABELS[n]:22s} mean f = {f_all.mean():.3f}   f > 0.9: {100 * np.mean(f_all > 0.9):4.1f}%   f < 0.1: {100 * np.mean(f_all < 0.1):4.1f}%")
plt.xlabel("forget gate value f"); plt.ylabel("count"); plt.title("Forget gates after training (1 = keep everything)")
plt.legend(fontsize=8); plt.show()

In [ ]:
#@title 📸 Save the combined figure → result.png
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
plot_val_curves(axes[0]); plot_test_bars(axes[1]); plot_grad(axes[2], "trained")
fig.suptitle(f"Read the vibe: RNN vs LSTM on {DATASET}" + ("  [QUICK]" if QUICK else ""), fontweight="bold")
plt.tight_layout()
plt.savefig("result.png", dpi=100, bbox_inches="tight")
plt.show()
import os
print(f"saved result.png ({os.path.getsize('result.png') / 1024:.0f} KB) | 🏆 {LABELS[best_name]}: {best_test_acc:.1f}% test accuracy")

## 💬 Discussion

- **RNN vs LSTM:** the RNN multiplies the gradient by `diag(1 − h²)·Wh` every step, the LSTM's cell state only by `f`. Look at your gradient plot: how many tokens back does the red curve fall 1000× below the LSTM curves?
- **imdb** is where this really bites: ~200 tokens, and the sentiment clue ("…but honestly, a masterpiece") can be anywhere. For short comments and tweets (≤ 40 tokens) the RNN often keeps up much better. Try both and compare!
- **Forget bias:** did `lstm_fb1` learn faster in the first epochs? Did it end up better, or did the default LSTM catch up? (Often the trick matters most early and on long texts.)
- **Forget gates after training:** is the mean `f` close to 1? How many long-memory slots (f > 0.9) did each LSTM grow? Does that line up with how far back the gradient reaches?

🤔 **Think about it:** the LSTM has 4× the recurrent weights of the RNN. Is the win just "more parameters"? (Hint: the gradient plot at **init**, before any learning, already shows the difference.)

🔜 **Next up: encoder-decoder.** An LSTM that reads a whole sentence ends with a final `(h, c)`, a summary of everything it read. Hand that summary to a *second* LSTM that writes a new sentence word by word and you've got an **encoder-decoder** (seq2seq), the model behind early neural translation. Same cell you just built, used twice.

## 🎤 Bonus: try your own sentence

Type anything (a text to your friend, a tweet draft, a hot take on a movie) and see what each model thinks.

In [ ]:
#@title 🎤 What's the vibe?
MY_TEXT = "i can't believe they cancelled my favourite show, this is the worst"  #@param {type:"string"}

@torch.no_grad()
def predict(text, model, k=3):
    model.eval()
    ids = encode(text)
    x = torch.tensor([ids], device=device)
    probs = F.softmax(model(x, torch.tensor([len(ids)])), dim=1)[0].cpu()
    top = probs.topk(min(k, NUM_CLASSES))
    return [(LABEL_NAMES[i], p.item()) for p, i in zip(top.values, top.indices)]

ids = encode(MY_TEXT)
print(f"📝 {MY_TEXT!r}  ({len(ids)} tokens, {sum(i == UNK for i in ids)} unknown)\n")
for n in [best_name] + [n for n in names if n != best_name]:
    preds = predict(MY_TEXT, results[n]["model"])
    star = " 🏆" if n == best_name else ""
    print(f"{LABELS[n]:22s}{star}: " + "   ".join(f"{label} {100 * p:.0f}%" for label, p in preds))

## 🚀 Submit

Paste `PLATFORM_URL` and `SUBMIT_TOKEN` (from the challenge page on DL Lab) into the settings cell at the top, run it, then run the cell below. It sends your best test accuracy and `result.png`.

In [ ]:
#@title 🚀 Submit your result to DL Lab
import base64, json, urllib.request, urllib.error
CHALLENGE_ID = "sequences"   # "sequences" | "encdec"

def submit(metric_name=None, metric=None, image_path=None, notes=""):
    if not PLATFORM_URL or not SUBMIT_TOKEN:
        print("⚠️ Paste PLATFORM_URL and SUBMIT_TOKEN into the settings cell at the top (copy them from the platform's challenge page).")
        return
    body = {"dataset": DATASET, "metric_name": metric_name, "metric": None if metric is None else float(metric), "notes": notes}
    if image_path:
        with open(image_path, "rb") as f:
            body["image_b64"] = base64.b64encode(f.read()).decode()
    req = urllib.request.Request(PLATFORM_URL.rstrip("/") + f"/api/challenges/{CHALLENGE_ID}/submit",
                                 data=json.dumps(body).encode(), method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": "Bearer " + SUBMIT_TOKEN.strip()})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            print("🎉", json.loads(r.read())["message"])
    except urllib.error.HTTPError as e:
        print("❌ Submit failed:", e.code, e.read().decode()[:300])
    except Exception as e:
        print("❌ Could not reach the platform:", e)

submit(metric_name="test_accuracy", metric=best_test_acc, image_path="result.png", notes=f"best={best_name}")